# Preprocessing Teks Ulasan E-commerce Berbahasa Indonesia

**Nama:** Muhammad Hafizh  
**NIM:** 24146028  
**Mata kuliah:** Natural Language Processing (NLP)

Notebook ini memandu langkah demi langkah preprocessing teks ulasan e-commerce
berbahasa Indonesia, mulai dari memuat dataset sampai membandingkan statistik
teks sebelum dan sesudah preprocessing.

**Dataset:** `dipawidia/ecommerce-product-reviews-sentiment`  
**Label:** `0` = negatif, `1` = positif  
**Split yang dipakai:** `train`

## Cara Menjalankan (Google Colab / Lokal)

**Di Google Colab:**
1. Upload file `.ipynb` ini, lalu pilih **Runtime -> Restart session** agar sesi bersih.
2. Jalankan sel **Persiapan Library** di bawah (perintah instal sudah disiapkan).
3. Pilih **Runtime -> Run all**. Jalankan dari atas ke bawah; jangan membuka sel demo
   sebelum sel definisi fungsinya.
4. Jika Colab meminta restart setelah instal, pilih **Runtime -> Restart session**,
   lalu **Run all** lagi.

**Di komputer lokal:** buka dengan Jupyter/VS Code, pilih kernel yang benar, lalu
jalankan semua sel dari atas ke bawah.

> Notebook ini mengunduh dataset dari internet dan menjalankan stemming Sastrawi.
> Proses stemming pada seluruh data bisa memakan beberapa menit.

## Persiapan Library

Sel ini memasang library yang dibutuhkan ke environment kernel yang sedang aktif.
`%pip` dipakai (bukan `!pip`) supaya paket pasti terpasang di Python yang dipakai
notebook, termasuk di Google Colab.

In [ ]:
# Memasang library yang belum tersedia (aman dijalankan berulang)
%pip install -q datasets Sastrawi

# Mengecek bahwa library bisa diimpor
import datasets
import Sastrawi
import pandas
import matplotlib

print("datasets   :", datasets.__version__)
print("pandas     :", pandas.__version__)
print("matplotlib :", matplotlib.__version__)
print("Sastrawi   : siap")

> Jika muncul pesan **"You must restart the runtime"**, lakukan
> **Runtime -> Restart session**, lalu jalankan lagi dari sel ini. Ini normal
> ketika Colab memperbarui library yang sudah terpasang.

## 1. Memuat Dataset

Tujuan langkah ini adalah mengunduh dataset dari HuggingFace dan mengubahnya
menjadi pandas DataFrame. DataFrame dipakai karena bentuknya seperti tabel
(baris = ulasan, kolom = atribut), sehingga lebih mudah diperiksa dan diolah.

In [ ]:
# Mengimpor library yang dibutuhkan
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

# Mengunduh dataset HuggingFace dan langsung mengambil split "train"
dataset = load_dataset("dipawidia/ecommerce-product-reviews-sentiment", split="train")

# Mengubah dataset menjadi pandas DataFrame
df = dataset.to_pandas()

# Menampilkan 5 baris pertama untuk memastikan data berhasil dimuat
df.head()

Sebelum memproses lebih jauh, nama kolom diperiksa langsung dari DataFrame.
Kolom teks dan label dideteksi dari kolom yang benar-benar ada, lalu divalidasi
agar tidak terjadi kesalahan nama kolom.

In [ ]:
print("Nama kolom :", df.columns.tolist())
print("Jumlah data:", len(df))

KOLOM_TEKS = next(
    (kolom for kolom in df.columns if kolom.lower() in {"review", "ulasan", "text", "teks", "komentar"}),
    None,
)
KOLOM_LABEL = next(
    (kolom for kolom in df.columns if kolom.lower() in {"sentimen", "sentiment", "label"}),
    None,
)

assert KOLOM_TEKS is not None, "Kolom teks ulasan tidak ditemukan."
assert KOLOM_LABEL is not None, "Kolom label sentimen tidak ditemukan."

print("Kolom teks  :", KOLOM_TEKS)
print("Kolom label :", KOLOM_LABEL)
print("Nilai label :", sorted(df[KOLOM_LABEL].dropna().unique().tolist()))

## 2. Informasi dan Distribusi Dataset

Sebelum mengolah teks, kita perlu tahu berapa banyak data dan bagaimana sebaran
label sentimennya. Informasi ini penting agar kita sadar apakah data seimbang;
jika sangat tidak seimbang, akurasi model bisa terlihat tinggi padahal model
hanya menebak kelas mayoritas. Grafik batang membantu melihat perbandingannya
secara cepat.

In [ ]:
# Menampilkan jumlah data, nama kolom, tipe data, missing value, dan duplikat
print("Jumlah data :", len(df))
print("Nama kolom  :", df.columns.tolist())
print("\nTipe data:")
print(df.dtypes)
print("\nTotal missing value :", int(df.isna().sum().sum()))
print("Jumlah baris duplikat:", int(df.duplicated().sum()))
print()

# Menghitung jumlah dan persentase tiap label sentimen
jumlah = df[KOLOM_LABEL].value_counts().sort_index()
persen = df[KOLOM_LABEL].value_counts(normalize=True).sort_index() * 100

# Merangkum distribusi label dalam tabel
ringkasan = pd.DataFrame({
    "Kode": jumlah.index,
    "Sentimen": ["Negatif" if kode == 0 else "Positif" for kode in jumlah.index],
    "Jumlah": jumlah.values,
    "Persentase (%)": persen.round(2).values,
})

ringkasan

In [ ]:
# Mengambil jumlah tiap label, diurutkan dari 0 ke 1
jumlah = df[KOLOM_LABEL].value_counts().sort_index()

# Membuat grafik batang
ax = jumlah.plot(kind="bar", color=["#e74c3c", "#2ecc71"])
ax.set_title("Distribusi Sentimen Ulasan")
ax.set_xlabel("Sentimen (0 = negatif, 1 = positif)")
ax.set_ylabel("Jumlah ulasan")
ax.set_xticklabels(["Negatif (0)", "Positif (1)"], rotation=0)

# Menulis angka jumlah di atas setiap batang
for i, nilai in enumerate(jumlah.values):
    ax.text(i, nilai + 80, str(nilai), ha="center")

plt.tight_layout()
plt.show()

## 3. Preprocessing Teks

Preprocessing adalah proses merapikan teks agar lebih seragam dan mudah
dianalisis. Kita memecahnya menjadi fungsi-fungsi kecil agar setiap tahap mudah
dipahami, diuji, dan diperbaiki. Urutannya penting: case folding dan cleaning
dilakukan sebelum tokenisasi, lalu stopword removal dan stemming pada level token.

> **Penting:** setiap subbab punya sel **definisi fungsi** lalu sel **demo**.
> Jalankan sel definisi terlebih dahulu. Jika memakai **Run all**, urutan ini
> sudah otomatis benar.

### 3.1 Case Folding

Case folding mengubah semua huruf menjadi huruf kecil. Tanpa langkah ini, kata
"Bagus" dan "bagus" akan dianggap sebagai dua kata yang berbeda, sehingga kosa
kata menjadi lebih besar dari yang seharusnya.

In [ ]:
def case_folding(teks):
    # Mengubah seluruh huruf di teks menjadi huruf kecil
    if not isinstance(teks, str):
        return ""
    return teks.lower()

In [ ]:
# Mengambil satu contoh ulasan
contoh = df.loc[0, KOLOM_TEKS]

print("Sebelum :", contoh)
print("Sesudah :", case_folding(contoh))

### 3.2 Cleaning

Cleaning membuang bagian teks yang tidak membantu analisis, seperti URL, angka,
tanda baca, dan karakter khusus. Tujuannya supaya yang tersisa hanya huruf dan
spasi, sehingga token yang dihasilkan lebih bersih dan konsisten.

In [ ]:
def cleaning(teks):
    # Menghapus URL, angka, tanda baca, dan karakter khusus
    teks = re.sub(r"http\S+|www\.\S+", " ", teks)  # hapus URL
    teks = re.sub(r"\d+", " ", teks)               # hapus angka
    teks = re.sub(r"[^a-z\s]", " ", teks)          # sisakan huruf a-z dan spasi
    teks = re.sub(r"\s+", " ", teks)               # rapikan spasi berlebih
    return teks.strip()

In [ ]:
contoh = case_folding(df.loc[0, KOLOM_TEKS])

print("Sebelum :", contoh)
print("Sesudah :", cleaning(contoh))

### 3.3 Tokenisasi

Tokenisasi memecah teks menjadi daftar kata (token). Tahap ini diperlukan karena
stopword removal dan stemming bekerja per kata, bukan pada kalimat utuh. Di sini
kita memakai pemisahan sederhana berdasarkan spasi.

In [ ]:
def tokenisasi(teks):
    # Memecah teks menjadi daftar kata berdasarkan spasi
    return teks.split()

In [ ]:
contoh = cleaning(case_folding(df.loc[0, KOLOM_TEKS]))
tokens = tokenisasi(contoh)

print("Teks  :", contoh)
print("Token :", tokens)
print("Jumlah token:", len(tokens))

### 3.4 Stopword Removal dengan Sastrawi

Stopword adalah kata umum yang sering muncul tetapi tidak banyak memberi makna,
misalnya "dan", "di", atau "yang". Sastrawi menyediakan daftar stopword bahasa
Indonesia, sehingga kita bisa membuang kata-kata tersebut dari daftar token.

In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Mengambil daftar stopword bahasa Indonesia dari Sastrawi
stopword_sastrawi = set(StopWordRemoverFactory().get_stop_words())

def hapus_stopword(tokens):
    # Membuang token yang termasuk stopword Sastrawi
    return [kata for kata in tokens if kata not in stopword_sastrawi]

print("Jumlah stopword Sastrawi:", len(stopword_sastrawi))

In [ ]:
tokens = tokenisasi(cleaning(case_folding(df.loc[0, KOLOM_TEKS])))

print("Sebelum :", tokens)
print("Sesudah :", hapus_stopword(tokens))

> **Catatan:** daftar stopword Sastrawi juga memuat kata negasi seperti "tidak".
> Untuk analisis sentimen, kata negasi sebaiknya dipertahankan karena bisa
> membalik makna kalimat. Di tugas ini kita tetap memakai daftar standar Sastrawi
> sesuai instruksi.

### 3.5 Stemming dengan Sastrawi dan Cache

Stemming mengubah kata berimbuhan menjadi kata dasarnya, misalnya "pengiriman"
menjadi "kirim". Proses ini berat jika setiap kemunculan kata dihitung ulang,
jadi kita menyimpan hasilnya di sebuah dictionary cache per kata unik. Dengan
begitu, kata yang sama hanya di-stem satu kali.

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Membuat objek stemmer Sastrawi
stemmer = StemmerFactory().create_stemmer()

# Cache: tempat menyimpan hasil stemming setiap kata unik
cache_stemming = {}

def stemming(tokens):
    # Mengubah token menjadi kata dasar menggunakan cache
    hasil = []

    for kata in tokens:
        if kata not in cache_stemming:                  # kata belum pernah di-stem
            cache_stemming[kata] = stemmer.stem(kata)   # hitung sekali saja
        hasil.append(cache_stemming[kata])              # ambil hasil dari cache

    return hasil

In [ ]:
tokens = hapus_stopword(tokenisasi(cleaning(case_folding(df.loc[0, KOLOM_TEKS]))))

print("Sebelum :", tokens)
print("Sesudah :", stemming(tokens))
print("Jumlah kata di cache:", len(cache_stemming))

### 3.6 Menggabungkan Semua Tahap

Sekarang kita gabungkan kelima fungsi di atas menjadi satu fungsi `preprocess`.
Dengan satu fungsi ini, seluruh tahap bisa dijalankan berurutan pada setiap
ulasan tanpa perlu menulis ulang kodenya.

In [ ]:
def preprocess(teks):
    # Menjalankan seluruh tahap preprocessing secara berurutan
    if not isinstance(teks, str):      # jaga-jaga jika ada data kosong
        return []

    teks = case_folding(teks)
    teks = cleaning(teks)
    tokens = tokenisasi(teks)
    tokens = hapus_stopword(tokens)
    tokens = stemming(tokens)

    return tokens

In [ ]:
contoh = df.loc[0, KOLOM_TEKS]

print("Sebelum :", contoh)
print("Sesudah :", " ".join(preprocess(contoh)))

### 3.7 Menerapkan Preprocessing ke Seluruh Data

Sel ini menjalankan `preprocess` pada semua ulasan. Hasil token disimpan di
kolom `tokens_bersih`, lalu digabung menjadi teks di kolom `teks_bersih`.
Kemajuan dicetak setiap 1.000 ulasan agar proses panjang tetap terpantau.

In [ ]:
# Menjalankan preprocessing pada seluruh ulasan
semua_tokens = []

for i, teks in enumerate(df[KOLOM_TEKS], start=1):
    semua_tokens.append(preprocess(teks))

    # Menampilkan progres setiap 1000 ulasan
    if i % 1000 == 0:
        print(f"{i} ulasan selesai diproses...")

# Menyimpan hasil ke DataFrame
df["tokens_bersih"] = pd.Series(semua_tokens, index=df.index, dtype="object")
df["teks_bersih"] = df["tokens_bersih"].apply(lambda token: " ".join(token))

print("Selesai! Total:", len(df), "ulasan")
print("Jumlah kata unik di cache stemming:", len(cache_stemming))
df[[KOLOM_TEKS, "teks_bersih"]].head()

Setelah preprocessing, sebagian ulasan bisa menjadi kosong (misalnya hanya berisi
angka, emoji, atau stopword). Baris seperti ini tidak dihapus, tetapi dihitung dan
ditampilkan agar transparan.

In [ ]:
kosong = df["teks_bersih"].str.strip().eq("") | df["teks_bersih"].isna()
print(f"Jumlah teks kosong setelah preprocessing: {int(kosong.sum())} dari {len(df)} baris "
      f"({kosong.mean() * 100:.2f}%)")

if kosong.any():
    df.loc[kosong, [KOLOM_TEKS, "teks_bersih"]].head(10)
else:
    print("Tidak ada teks yang kosong setelah preprocessing.")

## 4. Contoh Hasil Sebelum dan Sesudah Preprocessing

Bagian ini menampilkan 10 ulasan acak sebelum dan sesudah preprocessing dalam
satu tabel. Setiap tahap ditampilkan agar mudah diperiksa: teks asli, case
folding, cleaning, tokenisasi, stopword removal, dan stemming (hasil akhir).

In [ ]:
# Mengatur lebar kolom agar teks mudah dibaca
pd.set_option("display.max_colwidth", 100)

# Mengambil 10 ulasan acak agar contohnya lebih beragam
contoh_tabel = df.sample(10, random_state=42).copy()

# Menghitung ulang tiap tahap khusus untuk 10 contoh ini
contoh_tabel["case_folding"] = contoh_tabel[KOLOM_TEKS].apply(case_folding)
contoh_tabel["cleaning"] = contoh_tabel["case_folding"].apply(cleaning)
contoh_tabel["tokenisasi"] = contoh_tabel["cleaning"].apply(tokenisasi)
contoh_tabel["stopword_removal"] = contoh_tabel["tokenisasi"].apply(hapus_stopword)
contoh_tabel["stemming"] = contoh_tabel["stopword_removal"].apply(stemming)

# Token digabung kembali menjadi teks agar tabel mudah dibaca
for kolom in ["tokenisasi", "stopword_removal", "stemming"]:
    contoh_tabel[kolom] = contoh_tabel[kolom].apply(lambda token: " ".join(token))

# Mengubah kode sentimen menjadi nama yang mudah dibaca
contoh_tabel["label"] = contoh_tabel[KOLOM_LABEL].map({0: "Negatif", 1: "Positif"})

# Menyusun tabel akhir
tabel_contoh = contoh_tabel[[
    KOLOM_LABEL, "label", KOLOM_TEKS, "case_folding", "cleaning",
    "tokenisasi", "stopword_removal", "stemming",
]].rename(columns={
    KOLOM_LABEL: "Kode",
    "label": "Sentimen",
    KOLOM_TEKS: "Teks Asli",
    "case_folding": "Case Folding",
    "cleaning": "Cleaning",
    "tokenisasi": "Tokenisasi",
    "stopword_removal": "Stopword Removal",
    "stemming": "Stemming (Final)",
})

tabel_contoh.insert(0, "No", range(1, 11))
tabel_contoh

## 5. Statistik Sebelum vs Sesudah Preprocessing

Kita hitung rata-rata jumlah karakter, rata-rata jumlah kata, dan jumlah kosa
kata unik untuk melihat dampak preprocessing. Kosa kata unik adalah banyaknya kata
berbeda di seluruh dataset. Biasanya rata-rata jumlah kata sedikit berkurang,
sedangkan kosa kata unik turun lebih banyak karena variasi bentuk kata dan noise
sudah dihilangkan.

In [ ]:
# SEBELUM: teks asli dipecah menjadi kata (di-lowercase agar adil saat dihitung)
kata_sebelum = df[KOLOM_TEKS].str.lower().str.split()
rata_karakter_sebelum = df[KOLOM_TEKS].str.len().mean()
rata_kata_sebelum = kata_sebelum.apply(len).mean()
vocab_sebelum = set(kata for tokens in kata_sebelum for kata in tokens)

# SESUDAH: token hasil preprocessing
rata_karakter_sesudah = df["teks_bersih"].str.len().mean()
rata_kata_sesudah = df["tokens_bersih"].apply(len).mean()
vocab_sesudah = set(kata for tokens in df["tokens_bersih"] for kata in tokens)

# Merangkum perbandingan dalam tabel
statistik = pd.DataFrame({
    "Tahap": ["Sebelum preprocessing", "Sesudah preprocessing"],
    "Rata-rata karakter": [round(rata_karakter_sebelum, 2), round(rata_karakter_sesudah, 2)],
    "Rata-rata jumlah kata": [round(rata_kata_sebelum, 2), round(rata_kata_sesudah, 2)],
    "Jumlah kosa kata unik": [len(vocab_sebelum), len(vocab_sesudah)],
})

statistik

In [ ]:
# Menghitung besarnya penurunan kosa kata unik
selisih = len(vocab_sebelum) - len(vocab_sesudah)
persen_turun = selisih / len(vocab_sebelum) * 100

print(f"Kosa kata unik berkurang {selisih} kata ({persen_turun:.2f}%).")

### 5.1 Sepuluh Kata Paling Sering

Tabel berikut membandingkan kata yang paling sering muncul sebelum dan sesudah
stopword removal. Bagian "sebelum" memakai token hasil cleaning (sebelum stopword
dan stemming), sedangkan bagian "sesudah" memakai token final.

In [ ]:
# Token setelah cleaning, sebelum stopword removal dan stemming
tokens_cleaning = df[KOLOM_TEKS].apply(
    lambda teks: tokenisasi(cleaning(case_folding(teks)))
)

hitung_sebelum = Counter(kata for tokens in tokens_cleaning for kata in tokens)
hitung_sesudah = Counter(kata for tokens in df["tokens_bersih"] for kata in tokens)

top_sebelum = pd.DataFrame(hitung_sebelum.most_common(10), columns=["Kata", "Frekuensi"])
top_sesudah = pd.DataFrame(hitung_sesudah.most_common(10), columns=["Kata", "Frekuensi"])

top_banding = pd.DataFrame({
    "Peringkat": range(1, 11),
    "Sebelum: kata": top_sebelum["Kata"],
    "Sebelum: frekuensi": top_sebelum["Frekuensi"],
    "Sesudah: kata": top_sesudah["Kata"],
    "Sesudah: frekuensi": top_sesudah["Frekuensi"],
})

top_banding

> **Catatan:** pada teks asli, tanda baca masih menempel pada kata, sehingga
> jumlah kosa kata "sebelum" terlihat lebih besar. Perbandingan ini tetap berguna
> untuk menunjukkan efek pembersihan dan penyeragaman bentuk kata.

## Kesimpulan

Preprocessing berhasil membuat teks lebih seragam: huruf disamakan, noise
dibuang, kata umum dihilangkan, dan kata berimbuhan dikembalikan ke bentuk dasar.
Hasilnya, rata-rata jumlah kata dan kosa kata unik menurun, sehingga data lebih
ringkas dan siap dipakai untuk tahap berikutnya seperti TF-IDF dan klasifikasi
sentimen.

Beberapa risiko yang perlu diperhatikan:

- **Stopword removal** ikut membuang kata negasi seperti "tidak" dan "bukan",
  sehingga makna kalimat dapat berbalik (misalnya "tidak bagus" menjadi "bagus").
  Untuk analisis sentimen, kata negasi sebaiknya dikecualikan dari daftar stopword.
- **Cleaning** membuang angka dan emoji yang kadang membawa informasi penting,
  seperti jumlah pembelian atau ekspresi kepuasan.
- **Stemming** dapat salah mereduksi kata tidak baku, singkatan, atau nama produk,
  sehingga makna asli dapat bergeser.
- Variasi slang dan pengulangan huruf (misalnya "bagusss") belum sepenuhnya
  dinormalisasi oleh case folding dan stemming.

## 3 Pertanyaan Latihan

1. Mengapa case folding sebaiknya dilakukan sebelum cleaning dan tokenisasi?
   Apa yang bisa salah jika urutannya dibalik?
2. Stopword removal Sastrawi ikut membuang kata negasi seperti "tidak" dan
   "bukan". Jelaskan dampaknya untuk analisis sentimen, lalu usulkan solusi.
3. Lihat tabel statistik. Mengapa jumlah kosa kata unik turun jauh lebih besar
   dibandingkan rata-rata jumlah kata per ulasan? Kaitkan jawabanmu dengan
   peran stemming dan stopword removal.